# Remove Null values and Deduplication

## 1.Merge Deduplicated Customer Records into Silver Layer

In [0]:
MERGE INTO silver.customers AS target
USING (
  SELECT *
  FROM (
    SELECT
      cst_id,
      cst_key,
      TRIM(cst_firstname) AS cst_firstname,
      TRIM(cst_lastname) AS cst_lastname,
      TRIM(cst_marital_status) AS cst_marital_status,
      TRIM(cst_gndr) AS cst_gndr,
      cst_create_date,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS rn
    FROM bronze.customers
    WHERE cst_id IS NOT NULL AND cst_key IS NOT NULL AND cst_create_date IS NOT NULL
  ) AS deduped
  WHERE rn = 1
) AS source
ON target.cst_id = source.cst_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;



In [0]:
SELECT COUNT(*) FROM silver.customers;

In [0]:
SELECT * FROM silver.customers LIMIT 10;

## 2.Merge Deduplicated Products Records into Silver Layer

In [0]:
MERGE INTO silver.products AS target
USING (
  SELECT *
  FROM (
    SELECT
      prd_id,
      REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,               -- Extracted category ID
      SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,                      -- Transformed product key
      TRIM(prd_nm) AS prd_nm,
      prd_cost,
      TRIM(prd_line) AS prd_line,
      prd_start_dt,
      prd_end_dt,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY prd_id ORDER BY created_at DESC) AS rn
    FROM bronze.products
    WHERE 
      prd_id IS NOT NULL AND
      prd_key IS NOT NULL AND
      prd_nm IS NOT NULL AND
      prd_start_dt IS NOT NULL
  ) AS deduped
  WHERE rn = 1
) AS source
ON target.prd_id = source.prd_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;



## 3.Merge Deduplicated sales Records into Silver Layer

In [0]:
MERGE INTO silver.sales AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(sls_ord_num)   AS sls_ord_num,
      TRIM(sls_prod_key)  AS sls_prod_key,
      sls_cust_id,
      sls_order_dt,
      sls_ship_dt,
      sls_due_dt,
      sls_sales,
      sls_quantity,
      sls_price,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY sls_ord_num ORDER BY created_at DESC) AS rn
    FROM bronze.sales
    WHERE 
      sls_ord_num IS NOT NULL AND
      sls_prod_key IS NOT NULL AND
      sls_cust_id IS NOT NULL AND
      sls_order_dt IS NOT NULL AND
      sls_ship_dt IS NOT NULL AND
      sls_due_dt IS NOT NULL AND
      sls_quantity IS NOT NULL
  ) AS deduped
  WHERE rn = 1
) AS source
ON target.sls_ord_num = source.sls_ord_num  
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;

## 4.Merge Deduplicated Customer_erp Records into Silver Layer

In [0]:
MERGE INTO silver.customers_erp AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(cid) AS cid,
      bdate,
      TRIM(gen) AS gen,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY cid ORDER BY created_at DESC) AS rn
    FROM bronze.customers_erp
    WHERE cid IS NOT NULL
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.cid = source.cid
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;

## 5.Merge Deduplicated Location Records into Silver Layer

In [0]:
MERGE INTO silver.location AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(cid) AS cid,
      TRIM(cntry) AS cntry,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY cid ORDER BY created_at DESC) AS rn
    FROM bronze.location
    WHERE cid IS NOT NULL
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.cid = source.cid
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;

## 6.Merge Deduplicated Category Records into Silver Layer

In [0]:
MERGE INTO silver.category AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(id) AS id,
      TRIM(cat) AS cat,
      TRIM(subcat) AS subcat,
      TRIM(maintenance) AS maintenance,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) AS rn
    FROM bronze.category
    WHERE 
      id IS NOT NULL AND
      cat IS NOT NULL AND
      subcat IS NOT NULL AND
      maintenance IS NOT NULL
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.id = source.id
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;